<img src="logo.png" alt="Vegeta" width="240">

# The ducted fan (EDF) — efficiency, CFD, noise, vibration, stages (25b)

An electric ducted fan in the style of RC jets — a small rotor with many blades turning fast in a tight duct, stator vanes
behind it that take the swirl out, a tail cone and a nozzle — sized for the same job as notebook 25's propellers: the same
3 kg drone, the same thrust at cruise, climb and take-off (plus the drag of its own nacelle).

**This notebook computes only the fan.** The two- and three-blade propellers it is compared with come from notebook 25's
export (`designs/data/air_propeller_results.json`, its section 13); if that file is not there, the comparisons are left out
and the fan stands alone. The design inputs (section, aircraft, flight points, material) are read from the same file so both
notebooks describe the same aircraft; without it they are the defaults below.

```
1  the EDF: design (edit here), the duct model (boreas.ducted), the operating points
2  CAD: rotor with spinner; duct, stators, centre body
3  efficiency — in level flight, and against rpm at five airspeeds              (against the propellers, if exported)
4  power and endurance on the same battery                                      (")
5  acceleration — thrust available, top speed, climb                            (")
6  CFD of the EDF (rotor_mrf_installed) at low / medium / high rpm, and a ball movie
7  noise — rotor–stator interaction tones, the stator count, broadband, the sound (")
8  vibration — the fan blade in FEA, modes, Campbell, fatigue                    (")
9  side by side                                                                  (")
10 multi-stage fans — two and three fan rings
11 the fan maps for the flight models (MERLIN)
```

The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them; `AIR_PROP_CFD=smoke|full` as in notebook 25).

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil, sys, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
from tqdm.auto import tqdm
from IPython.display import Video, Audio, display
from vegeta import dedalus, talos, aeromant, boreas
from vegeta.boreas import ducted, fan_noise
from vegeta.aeromant import movie
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.dedalus.examples import Propeller as PropellerCAD
sys.path.insert(0, "designs")
import air_propeller as ap
import ducted_fan as df

RUNS = Path("_runs/ducted_fan"); shutil.rmtree(RUNS, ignore_errors=True); (RUNS / "cad").mkdir(parents=True)
OUT = Path("output/25b_ducted_fan"); OUT.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
CFD_PRESET = os.environ.get("AIR_PROP_CFD", "smoke")
PROCS = int(os.environ.get("AIR_PROP_PROCS", "4"))
RHO, NU, G = 1.225, 1.5e-5, 9.81
AIR = boreas.Medium("air", RHO, 340.0)

# notebook 25's export: the propellers to compare with (None: the fan alone)
RESULTS_JSON = Path("designs/data/air_propeller_results.json")
PJ = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else None
COMPARE = PJ is not None
print(f"propellers to compare with: {'notebook 25 export, ' + ', '.join(c['label'] for c in PJ['configs'].values()) if COMPARE else 'none (run notebook 25 to export them)'}")
print(f"CFD: {'on' if RUN_CFD else 'skipped'}, preset {CFD_PRESET!r}")

## 1. The EDF — edit here

The aircraft and the flight points are notebook 25's (from its export when there is one). The duct's losses are fitted to a
published static point of a 90 mm, 12-blade 6S hobby EDF (a seller's figure, not a measurement of ours): 3.0 kgf from 1930 W
electrical at an assumed 85 % motor + ESC efficiency — replace it with the fan you buy.

In [ ]:
SECTION = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-3.0, cl_max=1.1, cd0=0.018, k=0.04,
                         source="assumed for a small blade at Re ~ 1e5 (notebook 25)")
AIRCRAFT = dict(mass_kg=3.0, wing_area_m2=0.50, cd0=0.035, aspect_ratio=8.0, oswald=0.8)
POINTS = {"cruise": (20.0, 0.0), "climb": (14.0, 3.0), "take-off": (0.0, 0.0)}
MATERIAL = dict(name="PA6-GF30 (moulded)", E=8000.0, nu=0.35, rho=1.35e-9, yield_=100.0, ultimate=150.0, fatigue_1e7=30.0, damping=0.02)
SPEEDS = (10.0, 15.0, 20.0, 25.0, 30.0)
DIST, LISTEN_ANGLE, ANGLES, RATE = 10.0, 45.0, np.arange(10, 171, 5), 44100
if COMPARE:                                     # the same aircraft as notebook 25
    SECTION = boreas.Airfoil(**PJ["section"])
    AIRCRAFT, MATERIAL, SPEEDS = PJ["aircraft"], PJ["material"], tuple(PJ["speeds"])
    POINTS = {k: tuple(v) for k, v in PJ["points"].items()}
    ns = PJ["noise_setup"]
    DIST, LISTEN_ANGLE = ns["distance_m"] or DIST, ns["listen_angle_deg"] or LISTEN_ANGLE
    ANGLES = np.array(ns["angles_deg"]) if ns["angles_deg"] else ANGLES
SPEED_COLORS = ("#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281")
UNIT_COLOR = {"2 blades": "#1f77b4", "3 blades": "#9467bd", "EDF": "#d62728"}   # one colour per blade count; the ducted fan in red
PROP_NAMES = ["2 blades", "3 blades"] if COMPARE else []
BLADE_MAT = talos.Material(MATERIAL["name"], youngs_modulus=MATERIAL["E"], poissons_ratio=MATERIAL["nu"], density=MATERIAL["rho"],
                           yield_strength=MATERIAL["yield_"], source="assumed moulded glass-filled nylon")

def required_thrust(V, climb_rate=0.0):
    """Thrust the aircraft needs at airspeed V [m/s] (level flight or a steady climb): drag polar + weight component."""
    a = AIRCRAFT
    W = a["mass_kg"] * G
    if V <= 0:
        return 0.75 * W
    q = 0.5 * RHO * V**2
    cl = W / (q * a["wing_area_m2"])
    k = 1 / (math.pi * a["aspect_ratio"] * a["oswald"])
    return q * a["wing_area_m2"] * (a["cd0"] + k * cl**2) + W * climb_rate / V

def prop(name):
    """notebook 25's export for '2 blades' / '3 blades'."""
    return PJ["propellers"][name.split()[0]]

def prop_op(name, point):
    return prop(name)["operating_points"][point]

EDF = dict(diameter_mm=90.0, blades=12, hub_diameter_mm=40.0, hub_height_mm=14.0, pitch_mm=160.0,
           chord_root_mm=14.0, chord_max_mm=15.0, chord_tip_mm=12.0, thickness=0.08, camber=0.04,
           tip_clearance_mm=0.8, exit_area_ratio=0.9, stator_vanes=7, stator_chord_mm=16.0, stator_thickness=0.08, stator_gap_mm=14.0,
           stator_loss=0.10, unit_mass_kg=0.32)                          # mass: fan unit with motor (assumed, for the comparison)
# the duct's losses are fitted to a published static point of a 90 mm, 12-blade 6S hobby EDF (a seller's figure, not a measurement
# of ours): 3.0 kgf from 1930 W electrical, at an assumed 85 % motor + ESC efficiency — replace it with the fan you buy
EDF_STATIC_POINT = dict(thrust_N=3.0 * G, electrical_W=1930.0, drive_efficiency=0.85)
EDF_BLADE = boreas.Propeller.from_pitch(f"EDF {EDF['diameter_mm']:.0f} mm, {EDF['blades']} blades", EDF["diameter_mm"] / 1000, EDF["pitch_mm"] / 1000,
                                        blades=EDF["blades"], chord_root_m=EDF["chord_root_mm"] / 1000, chord_max_m=EDF["chord_max_mm"] / 1000,
                                        chord_tip_m=EDF["chord_tip_mm"] / 1000, hub_radius_m=EDF["hub_diameter_mm"] / 2000)
ROTOR_P = dict(diameter=EDF["diameter_mm"], hub_diameter=EDF["hub_diameter_mm"], hub_height=EDF["hub_height_mm"], pitch=EDF["pitch_mm"],
               chord_root=EDF["chord_root_mm"], chord_max=EDF["chord_max_mm"], chord_tip=EDF["chord_tip_mm"], thickness=EDF["thickness"],
               camber=EDF["camber"])
HOUSING_P = dict(ROTOR_P, tip_clearance=EDF["tip_clearance_mm"], exit_area_ratio=EDF["exit_area_ratio"], stator_vanes=EDF["stator_vanes"],
                 axial_gap=2.0,                                          # rotor hub to the motor housing: keeps the blade roots clear for the mesher
                 stator_chord=EDF["stator_chord_mm"], stator_thickness=EDF["stator_thickness"], stator_gap=EDF["stator_gap_mm"])
HG = df.housing_geometry(HOUSING_P)
FAN = ducted.DuctedFan(EDF_BLADE.name, EDF_BLADE, tip_clearance_m=EDF["tip_clearance_mm"] / 1000, exit_area_ratio=HG["exit_area_ratio"],
                       stator_vanes=EDF["stator_vanes"], stator_loss=EDF["stator_loss"],
                       external_wetted_area_m2=HG["external_wetted_area"] * 1e-6, duct_length_m=HG["total_length"] / 1000, mass_kg=EDF["unit_mass_kg"])
FAN = ducted.fit_duct_loss(FAN, SECTION, EDF_STATIC_POINT["thrust_N"], EDF_STATIC_POINT["electrical_W"] * EDF_STATIC_POINT["drive_efficiency"], 0.0, RHO)
print(f"duct_loss fitted to the static point: {FAN.duct_loss:.3f}")
EDF_DUCT_R = HG["shroud_radius"] / 1000                                   # for the duct's cut-off of the interaction modes
EDF_DUCT_L = (HG["exit_x"] - HG["stator_te_x"]) / 1000
display(pd.Series({k: v for k, v in HG.items() if np.isscalar(v)}, name="EDF housing (mm, mm²)").to_frame().round(2))

def edf_point(V, need):
    """The EDF at airspeed V making the net thrust `need` (its nacelle's friction drag added on top)."""
    return ducted.rpm_for_thrust(FAN, SECTION, need + ducted.nacelle_drag(FAN, V, RHO, NU), V, RHO)

EDF_OPS = {k: edf_point(V, required_thrust(V, cr)) for k, (V, cr) in POINTS.items()}
rows_ = {}
for k in POINTS:
    for name in PROP_NAMES:
        op = prop_op(name, k)
        rows_[(k, name)] = {"rpm": op["rpm"], "thrust [N]": op["thrust"], "shaft power [W]": op["power"], "η": op["efficiency"],
                            "figure of merit": op["figure_of_merit"], "tip Mach": op["tip_mach"]}
    e = EDF_OPS[k]
    rows_[(k, "EDF")] = {"rpm": e.rpm, "thrust [N]": e.thrust, "shaft power [W]": e.power, "η": e.efficiency,
                         "figure of merit": e.figure_of_merit, "tip Mach": e.tip_mach, "jet speed [m/s]": e.exit_velocity,
                         "rotor / duct thrust [N]": f"{e.rotor_thrust:.2f} / {e.duct_thrust:.2f}"}
print(f"EDF static figure of merit at take-off {EDF_OPS['take-off'].figure_of_merit:.2f} (duct_loss {FAN.duct_loss:.3f}, fitted above)")
pd.DataFrame(rows_).T

## 2. CAD

In [ ]:
rotor_cad = df.EDFRotor().generate(blades=EDF["blades"], **ROTOR_P)
rotor_x = dedalus.Geometry.from_cadquery(rotor_cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="edf_rotor_axis_x")
EDF_ROTOR_STL = rotor_x.export_stl(RUNS / "cad" / "edf_rotor_axis_x.stl", tolerance=0.02)
housing_cad = df.EDFHousing().generate(**HOUSING_P)
EDF_HOUSING_STL = housing_cad.export(RUNS / "cad" / "edf_housing", formats=("stl", "step"), stl_tolerance=0.05).artifacts["stl"]
EDF_ROTOR_MESH = dviz.to_pyvista(rotor_x, 0.03).scale(0.001, inplace=False)
EDF_HOUSING_MESH = dviz.to_pyvista(housing_cad, 0.05).scale(0.001, inplace=False)
EDF_OUTLINE = df.outline(HOUSING_P)
pl = pv.Plotter(shape=(1, 2), window_size=(1400, 520))
pl.subplot(0, 0)
pl.add_mesh(EDF_HOUSING_MESH, color="#c9c9c9", opacity=0.35, smooth_shading=True)
pl.add_mesh(EDF_ROTOR_MESH, color="#3b6ea5", smooth_shading=True)
pl.add_text("EDF: rotor inside the housing (housing see-through)", font_size=10)
pl.camera_position = [(-0.16, 0.10, 0.12), (0.01, 0.0, 0.0), (0, 1, 0)]
pl.subplot(0, 1)
pl.add_mesh(EDF_HOUSING_MESH.clip(normal="z", origin=(0, 0, 0)), color="#c9c9c9", smooth_shading=True)
pl.add_mesh(EDF_ROTOR_MESH.clip(normal="z", origin=(0, 0, 0)), color="#3b6ea5", smooth_shading=True)
pl.add_text("cut through the axis: lip, rotor, stators, centre body, nozzle", font_size=10)
pl.camera_position = [(0.02, 0.0, 0.30), (0.02, 0.0, 0.0), (0, 1, 0)]
aviz.show(pl)

## 3. Efficiency

Left: the efficiency at the thrust the aircraft needs in level flight, each unit at its own rpm (the EDF net of its nacelle's
drag). Right: the EDF's efficiency against rpm at five airspeeds (and the propellers' alone, from notebook 25). A duct
recovers the tip loss and its lip makes thrust, so a ducted fan is the better **static** thruster for its size; but it throws
a small, fast jet: its propulsive efficiency `2V / (V + V_jet)` is low at a drone's speeds.

In [ ]:
VS = np.linspace(9.0, 32.0, 24)
ETA_LEVEL = {name: np.interp(VS, prop(name)["level"]["V"], prop(name)["level"]["eta"]) for name in PROP_NAMES}
ETA_LEVEL["EDF"] = np.array([(lambda e: (e.thrust - ducted.nacelle_drag(FAN, V, RHO, NU)) * V / e.power)(edf_point(V, required_thrust(V))) for V in VS])
EDF_RPM_MAX = 1.15 * EDF_OPS["take-off"].rpm

def edf_curve(V, n=40):
    lo, hi = 1000.0, EDF_RPM_MAX
    for _ in range(22):                                         # zero (net) thrust rpm at V
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if ducted.solve(FAN, SECTION, mid, V, RHO).thrust <= ducted.nacelle_drag(FAN, V, RHO, NU) else (lo, mid)
    out = []
    for n_ in hi + (EDF_RPM_MAX - hi) * np.linspace(0, 1, n) ** 1.8:
        e = ducted.solve(FAN, SECTION, n_, V, RHO)
        net = e.thrust - ducted.nacelle_drag(FAN, V, RHO, NU)
        out.append((n_, net * V / e.power if net > 0 and e.power > 0 else np.nan))
    return np.array(out)

EDF_CURVES = {V: edf_curve(V) for V in SPEEDS}
panels = ["level"] + PROP_NAMES + ["EDF"]
fig, axs = plt.subplots(1, len(panels), figsize=(5.4 * len(panels), 4.6))
for name, eta in ETA_LEVEL.items():
    axs[0].plot(VS, eta, color=UNIT_COLOR[name], lw=2, label=name + (" (10 × 6, notebook 25)" if name != "EDF" else f" ({EDF['diameter_mm']:.0f} mm, {EDF['blades']} blades)"))
axs[0].plot(VS, [2 * V / (V + edf_point(V, required_thrust(V)).exit_velocity) for V in VS], ":", color=UNIT_COLOR["EDF"], lw=1.5,
            label="EDF: ideal propulsive limit 2V/(V+V_jet)")
axs[0].set(xlabel="airspeed [m/s]", ylabel="efficiency (net thrust × V / shaft power)", title="in level flight, at the thrust the aircraft needs", ylim=(0, 0.9))
axs[0].legend(fontsize=8)
for ax, name in zip(axs[1:], panels[1:]):
    for V, col in zip(SPEEDS, SPEED_COLORS):
        if name == "EDF":
            ax.plot(EDF_CURVES[V][:, 0], EDF_CURVES[V][:, 1], color=col, lw=2, label=f"{V:.0f} m/s")
        else:
            cur = np.array(prop(name)["eta_vs_rpm_alone"][f"{V:g}"], float)
            ax.plot(cur[:, 0], cur[:, 1], color=col, lw=2, label=f"{V:.0f} m/s")
    ax.set(xlabel="rpm", title=(f"EDF {EDF['diameter_mm']:.0f} mm, {EDF['blades']} blades (net of the nacelle)" if name == "EDF" else f"10 × 6, {name}, alone (notebook 25)"),
           ylim=(0, 0.85))
for ax in axs:
    ax.grid(color="#e8e8e8", lw=0.6)
axs[-1].legend(fontsize=8, title="airspeed")
fig.tight_layout()
pd.DataFrame({name: {f"η at {V:.0f} m/s": float(np.interp(V, VS, eta)) for V in (10.0, 15.0, 20.0, 25.0, 30.0)} for name, eta in ETA_LEVEL.items()}).round(3)

## 4. Power and endurance on the same battery

Shaft power in level flight against airspeed, and the endurance and range at cruise on the same battery (4S 5000 mAh, 74 Wh,
80 % usable) through the same motor and ESC efficiency (0.85, assumed for all — a fast EDF motor is often worse).

In [ ]:
BATTERY_WH, USABLE, DRIVE_EFF = 74.0, 0.80, 0.85
P_LEVEL = {name: np.interp(VS, prop(name)["level"]["V"], prop(name)["level"]["power_W"]) for name in PROP_NAMES}
P_LEVEL["EDF"] = np.array([edf_point(V, required_thrust(V)).power for V in VS])
fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
for name, P in P_LEVEL.items():
    ax[0].plot(VS, P, color=UNIT_COLOR[name], lw=2, label=name)
ax[0].set(xlabel="airspeed [m/s]", ylabel="shaft power [W]", title="power in level flight"); ax[0].legend(); ax[0].grid(color="#e8e8e8", lw=0.6)
POWER = {}
for name in PROP_NAMES + ["EDF"]:
    pw = {k: (EDF_OPS[k].power if name == "EDF" else prop_op(name, k)["power"]) for k in POINTS}
    p_c = pw["cruise"] / DRIVE_EFF
    POWER[name] = {"cruise shaft power [W]": pw["cruise"], "climb shaft power [W]": pw["climb"], "take-off shaft power [W]": pw["take-off"],
                   "cruise battery power [W]": p_c, "endurance at cruise [min]": BATTERY_WH * USABLE / p_c * 60,
                   "range at cruise [km]": BATTERY_WH * USABLE / p_c * POINTS["cruise"][0] * 3.6}
POWER = pd.DataFrame(POWER)
x = np.arange(3)
w_ = 0.8 / len(POWER.columns)
for i, name in enumerate(POWER.columns):
    ax[1].bar(x + (i - (len(POWER.columns) - 1) / 2) * w_, POWER.loc[["cruise shaft power [W]", "climb shaft power [W]", "take-off shaft power [W]"], name], w_ * 0.95,
              color=UNIT_COLOR[name], label=name)
ax[1].set_xticks(x, ["cruise", "climb", "take-off"]); ax[1].set(ylabel="shaft power [W]", title="by flight point"); ax[1].legend(); ax[1].grid(color="#e8e8e8", lw=0.6, axis="y")
fig.tight_layout()
POWER.round(1)

## 5. Acceleration

Full throttle is each unit's take-off rpm, held: the thrust against airspeed, minus the aircraft's drag, over the mass is the
acceleration. From it the time from a hand launch at 9 m/s to cruise, the top speed in level flight and the best climb.

In [ ]:
PROP_MASS = 0.045                                                         # kg: 10x6 propeller (for the unit-mass difference)
def full_thrust(name, V):
    if name == "EDF":
        e = ducted.solve(FAN, SECTION, EDF_OPS["take-off"].rpm, V, RHO)
        return e.thrust - ducted.nacelle_drag(FAN, V, RHO, NU), e.power
    ft = prop(name)["full_throttle"]
    return float(np.interp(V, ft["V"], ft["thrust_N"])), float(np.interp(V, ft["V"], ft["power_W"]))

def aircraft_mass(name):
    return AIRCRAFT["mass_kg"] + (EDF["unit_mass_kg"] - PROP_MASS - 0.11 if name == "EDF" else 0.0)   # 0.11 kg: the open prop's motor

VA = np.linspace(0.0, 40.0, 81)
ACC = {}
fig, ax = plt.subplots(1, 3, figsize=(19, 4.4))
for name in PROP_NAMES + ["EDF"]:
    T = np.array([full_thrust(name, V)[0] for V in VA])
    D = np.array([required_thrust(V) if V > 0 else 0.0 for V in VA])
    m = aircraft_mass(name)
    a = (T - D) / m
    ACC[name] = (T, D, a)
    ax[0].plot(VA, T, color=UNIT_COLOR[name], lw=2, label=f"{name}: thrust at full throttle")
    ax[1].plot(VA, a, color=UNIT_COLOR[name], lw=2, label=name)
ax[0].plot(VA[VA >= 8], [required_thrust(V) for V in VA[VA >= 8]], color="#222222", lw=1.5, label="aircraft drag, level flight")
ax[0].set(xlabel="airspeed [m/s]", ylabel="force [N]", title="thrust available and drag", ylim=(0, None)); ax[0].legend(fontsize=8)
ax[1].axhline(0, color="#222222", lw=0.8)
ax[1].set(xlabel="airspeed [m/s]", ylabel="acceleration [m/s²]", title="level acceleration at full throttle", xlim=(8, 40)); ax[1].legend()
ACC_TAB = {}
for name, (T, D, a) in ACC.items():
    ok = VA >= 9.0
    v_top = float(VA[ok][np.where(a[ok] <= 0)[0][0]]) if (a[ok] <= 0).any() else float(VA[-1])
    vv = np.linspace(9.0, POINTS["cruise"][0], 200)
    aa = np.interp(vv, VA, a)
    t_cruise = float(np.trapezoid(1 / np.maximum(aa, 1e-6), vv)) if (aa > 0).all() else np.inf
    m = aircraft_mass(name)
    climb = max((np.interp(V, VA, T) - required_thrust(V)) * V / (m * G) for V in np.linspace(10, 30, 41))
    ACC_TAB[name] = {"acceleration at 10 m/s [m/s²]": float(np.interp(10, VA, a)), "acceleration at 20 m/s [m/s²]": float(np.interp(20, VA, a)),
                     "time 9 → 20 m/s [s]": t_cruise, "top speed, level [m/s]": v_top, "best climb rate [m/s]": climb,
                     "static thrust [N]": float(T[0]), "mass [kg]": m}
    ts = np.concatenate([[0], np.cumsum(np.diff(vv) / np.maximum(0.5 * (aa[1:] + aa[:-1]), 1e-6))])
    ax[2].plot(ts, vv, color=UNIT_COLOR[name], lw=2, label=f"{name}: {t_cruise:.1f} s")
ax[2].set(xlabel="time from the hand launch [s]", ylabel="airspeed [m/s]", title="from a 9 m/s hand launch to cruise"); ax[2].legend()
for a_ in ax:
    a_.grid(color="#e8e8e8", lw=0.6)
fig.tight_layout()
ACC_TAB = pd.DataFrame(ACC_TAB)
ACC_TAB.round(2)

## 6. CFD of the EDF — low, medium, high rpm, and a ball movie

`rotor_mrf_installed`: the rotor with its spinner in the rotating zone (a cylinder just inside the shroud), the duct, stators
and centre body standing. The unit's net thrust is `net_thrust_N` (rotor thrust minus the housing's drag, negative when the
lip pulls). Three rpm points at the cruise speed; the smoke preset only proves that the case meshes and runs.

In [ ]:
RPM_POINTS = {"low": 0.85, "medium": 1.0, "high": 1.25}
PRESETS = {
    "smoke": dict(rotor=dict(iterations=60, cells_per_diameter=5.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1,
                             upstream=2.0, downstream=3.0, lateral=2.0), static=dict(static_level=2, static_box_level=1, static_margin=1.0)),
    "full": dict(rotor=dict(iterations=600, cells_per_diameter=8.0, surface_level=4, near_level=3, rotor_level=3, wake_level=2,
                            upstream=3.0, downstream=6.0, lateral=3.0), static=dict(static_level=3, static_box_level=1, static_margin=1.5)),
}
PRE = PRESETS[CFD_PRESET]
ENV = aeromant.OpenFOAMEnvironment.detect() if RUN_CFD else aeromant.OpenFOAMEnvironment()
V_CFD = POINTS["cruise"][0]
EDF_D = EDF["diameter_mm"] / 1000
EDF_ZONE = dict(zone_radius=(EDF["diameter_mm"] / 2 + 0.5 * EDF["tip_clearance_mm"]) / EDF["diameter_mm"],   # inside the clearance
                zone_length=0.36)                                                                         # ends before the stators
EDF_CASES = {}
for name, f in RPM_POINTS.items():
    rpm = f * EDF_OPS["cruise"].rpm
    EDF_CASES[name] = aeromant.CFDCase("rotor_mrf_installed", EDF_ROTOR_STL,
        dict(rpm=rpm, airspeed=V_CFD, diameter=EDF_D, kinematic_viscosity=NU, density=RHO, rotation=1, **PRE["rotor"], **PRE["static"], **EDF_ZONE),
        RUNS / f"cfd_edf_{name}_{rpm:.0f}rpm", geometry_units="mm", environment=ENV, static_geometry=EDF_HOUSING_STL)
    r = EDF_CASES[name].prepare(overwrite=True)
    assert r.ok, (name, r.messages)
EDF_CFD = {}
if RUN_CFD:
    for name, case in tqdm(EDF_CASES.items(), desc="EDF CFD"):
        EDF_CFD[name] = case.run(processors=PROCS)
        if not EDF_CFD[name].ok:
            print(name, "FAILED:", EDF_CFD[name].messages[:3])
EDF_CFD = {k: r for k, r in EDF_CFD.items() if r.ok}
rows_ = {}
for name, r in EDF_CFD.items():
    m = r.metrics
    rpm = m["omega_rad_s"] * 60 / (2 * math.pi)
    e = ducted.solve(FAN, SECTION, rpm, V_CFD, RHO)
    rows_[f"{name} ({rpm:.0f} rpm)"] = {"cells": m.get("mesh_cells"), "converged": m.get("converged"),
                                         "unit thrust, model [N]": e.thrust, "rotor thrust, model [N]": e.rotor_thrust,
                                         "rotor thrust, CFD [N]": m["thrust_N"], "housing force, CFD [N]": m.get("body_drag_N"),
                                         "unit net thrust, CFD [N]": m.get("net_thrust_N"), "power, model [W]": e.power, "power, CFD [W]": m["power_W"],
                                         "η model": e.efficiency, "η unit, CFD": m.get("net_efficiency")}
if CFD_PRESET == "smoke" and rows_:
    print("SMOKE TEST numbers (coarse, unconverged): the EDF case meshes and runs; these are NOT results")
aviz.show(aviz.plot_setup(EDF_CASES["medium"]))
pd.DataFrame(rows_).T.round(3) if rows_ else "EDF CFD not run"

In [ ]:
if EDF_CFD:
    key = "medium" if "medium" in EDF_CFD else next(iter(EDF_CFD))
    fig, ax = plt.subplots(figsize=(14, 4.6))
    aviz.plot_section(EDF_CASES[key], field="U", normal="z", origin=(0.0, 0.0, 0.0), ax=ax, levels=30)
    for poly in EDF_OUTLINE["side"]:
        ax.fill(poly[:, 0], poly[:, 1], color="#dddddd", ec="#666666", lw=0.8)
    ax.set_xlim(-0.12, 0.25); ax.set_ylim(-0.08, 0.08); ax.set_title(f"EDF, {key} rpm: |U| through the axis ({CFD_PRESET})")
    s_edf = movie.openfoam_sampler(EDF_CASES[key])
    r_edf = movie.RotorView.from_case(EDF_CASES[key], EDF["blades"])
    EDF_MOVIE = movie.make_movie(None, OUT / "flow_edf.mp4", rotor=r_edf, sampler=s_edf, n=60, seconds=5, fps=24, degrees_per_frame=10,
                                 outline=EDF_OUTLINE, view=dict(feed_upstream=1.6, feed_radius=0.45, downstream=3.0, lateral=0.9),
                                 title=f"EDF {EDF['diameter_mm']:.0f} mm: {V_CFD:.0f} m/s, {r_edf.rpm:.0f} rpm (OpenFOAM, {CFD_PRESET})")
    prop_clips = [(PJ["configs"][k]["flow_movie"], PJ["configs"][k]["label"]) for k in ("tractor 2", "pusher 2")
                  if COMPARE and PJ["configs"][k]["flow_movie"] and Path(PJ["configs"][k]["flow_movie"]).exists()]
    if prop_clips:                                   # notebook 25's propeller movies first, then the fan
        EDF_MOVIE_ALL = movie.concat_videos([Path(p) for p, _ in prop_clips] + [EDF_MOVIE], OUT / "flow_props_then_edf.mp4",
                                            captions=[f"open, {c}" for _, c in prop_clips] + ["ducted fan"])
        display(Video(str(EDF_MOVIE_ALL), embed=False, width=960))
    else:
        display(Video(str(EDF_MOVIE), embed=False, width=960))
else:
    print("no EDF CFD field: the EDF movie needs the CFD (run with OpenFOAM)")

## 7. Noise

Three sources at 10 m: the **rotor-alone tones** (cut off in the duct at a subsonic tip — shown free-field for reference),
the **rotor–stator interaction tones** (`boreas.fan_noise`: Silverstein wakes, the Sears response, Tyler–Sofrin modes —
the stator count decides which harmonics escape) and **broadband** (vortex noise on the rotor and the vanes). Twelve blades
at the EDF's rpm put the blade-passing tone where the ear and the A-weighting are most sensitive — the familiar EDF whine.
The propellers' levels are notebook 25's.

In [ ]:
def edf_noise(point="cruise", angle=LISTEN_ANGLE, distance=DIST, vanes=None, harmonics=4):
    e = EDF_OPS[point] if isinstance(point, str) else point
    V_ = int(vanes or EDF["stator_vanes"])
    omega = e.rpm * 2 * math.pi / 60
    r_m = 0.5 * (EDF_BLADE.hub_radius + EDF_BLADE.radius)
    c_th = float(np.interp(r_m, e.r, e.c_theta))
    cl_m = float(np.interp(r_m, e.r, e.cl))
    cd_m = SECTION.cd0 + SECTION.k * cl_m ** 2
    chord_m = float(np.interp(r_m, EDF_BLADE.r, EDF_BLADE.chord))
    beta_exit = math.degrees(math.atan2(omega * r_m - c_th, e.fan_velocity))   # the rotor wakes travel along the relative exit flow
    wake_h = fan_noise.rotor_wake_harmonics(EDF["blades"], chord_m, cd_m, EDF["stator_gap_mm"] / 1000, r_m, harmonics=harmonics,
                                            wake_angle_deg=beta_exit)
    gust = wake_h * (omega * r_m - c_th)                                 # tangential upwash on an axial vane: the wake deficit x (omega r - c_theta)
    U_v = math.hypot(e.fan_velocity, c_th)
    inter = fan_noise.interaction_tones(EDF["blades"], V_, e.rpm, vane_radius_m=r_m, vane_chord_m=EDF["stator_chord_mm"] / 1000,
                                        vane_span_m=EDF_BLADE.radius - EDF_BLADE.hub_radius, vane_inflow_m_s=U_v, gust_m_s=gust,
                                        distance=distance, medium=AIR, angle_deg=180.0 - angle,   # fan_noise: angle from the downstream axis
                                        azimuth_deg=0.0, stagger_deg=0.0, harmonics=harmonics,
                                        duct_radius_m=EDF_DUCT_R, duct_length_m=EDF_DUCT_L)
    alone = boreas.gutin_harmonics(EDF_BLADE, max(e.rotor_thrust, 0.0), e.torque, e.rpm, distance, angle, AIR, harmonics=harmonics)
    vn = boreas.vortex_noise(EDF_BLADE, e.rpm, distance, e.fan_velocity, thickness_ratio=EDF["thickness"])
    a_vanes = V_ * EDF["stator_chord_mm"] / 1000 * (EDF_BLADE.radius - EDF_BLADE.hub_radius)
    ft = 0.3048
    vanes_bb = 10 * math.log10(3.8e-27 * (a_vanes / ft ** 2) * (U_v / ft) ** 6 / 1e-16) + 20 * math.log10(300 * ft / distance)
    bb = boreas.add_levels([vn["spl_db"], vanes_bb])
    band = (vn["peak_hz"] / 2, min(vn["peak_hz"] * 2, 20000.0))
    bb_a = bb + 10 * math.log10(np.mean(10 ** (boreas.a_weighting(np.geomspace(*band, 200)) / 10)))
    f = np.array([t["frequency_hz"] for t in inter]); L = np.array([t["spl_ring_db"] for t in inter])   # azimuth-averaged
    tonal, tonal_a = boreas.add_levels(L), boreas.add_levels(L + boreas.a_weighting(f))
    return dict(tones=[(t["frequency_hz"], t["spl_ring_db"]) for t in inter], inter=inter, alone=alone, broadband=bb, band=band,
                tonal=tonal, tonal_A=tonal_a, total=boreas.add_levels([tonal, bb]), total_A=boreas.add_levels([tonal_a, bb_a]),
                bpf=EDF["blades"] * e.rpm / 60, rpm=e.rpm)

EDF_NOISE = edf_noise()
EDF_DIRECTIVITY = np.array([edf_noise(angle=a)["total_A"] for a in ANGLES])
fig = plt.figure(figsize=(19, 4.6))
ax = fig.add_subplot(1, 3, 1)
n_ = EDF_NOISE
ax.stem([t["frequency_hz"] for t in n_["inter"]], [t["spl_ring_db"] for t in n_["inter"]], linefmt=UNIT_COLOR["EDF"], markerfmt="o", basefmt=" ",
        label="rotor–stator interaction (azimuth-averaged)")
for t in n_["inter"]:
    ax.annotate(f"n = {t['dominant_n']}", (t["frequency_hz"], t["spl_ring_db"]), xytext=(4, 4), textcoords="offset points", fontsize=8, color="#555555")
ax.plot(n_["alone"]["frequency_hz"], n_["alone"]["spl_db"], "x", color="#222222", label="rotor alone (free field; cut off in the duct)")
ax.hlines(n_["broadband"] - 10 * math.log10(3 * math.log2(n_["band"][1] / n_["band"][0])), *n_["band"], color="#888888", ls="--", label="broadband (per third octave)")
ax.set(xscale="log", xlabel="frequency [Hz]", ylabel=f"dB re 20 µPa at {DIST:.0f} m", ylim=(0, 80),
       title=f"EDF at cruise, {LISTEN_ANGLE:.0f}° from the axis: BPF {n_['bpf']:.0f} Hz\n{n_['total']:.1f} dB, {n_['total_A']:.1f} dB(A)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
axp = fig.add_subplot(1, 3, 2, projection="polar")
PROP_DIR = {c["label"]: (np.array(c["directivity_dBA"]), {2: "#1f77b4", 3: "#9467bd"}[c["blades"]], "--" if c["layout"] == "tractor" else "-")
            for c in (PJ["configs"].values() if COMPARE else []) if c["directivity_dBA"]}
for lab, (d_, col, ls) in PROP_DIR.items():
    axp.plot(np.radians(ANGLES), d_, ls, color=col, lw=1.5, label=f"open {lab}")
axp.plot(np.radians(ANGLES), EDF_DIRECTIVITY, color=UNIT_COLOR["EDF"], lw=2.5, label="EDF")
axp.set_thetamin(0); axp.set_thetamax(180); axp.set_theta_zero_location("N"); axp.set_theta_direction(-1)
axp.set_title(f"directivity at cruise, dB(A) at {DIST:.0f} m"); axp.legend(fontsize=7, loc="lower left")
ax = fig.add_subplot(1, 3, 3)
VANES = (5, 7, 9, 11, 13, 17, 25)
study = fan_noise.vane_count_study(EDF["blades"], VANES, EDF_OPS["cruise"].tip_mach, harmonics=2)
lev = [max(edf_noise(vanes=v, angle=a)["tonal_A"] for a in (20, 45, 90, 135, 160)) for v in VANES]
ax.bar(range(len(VANES)), lev, color=["#d62728" if v == EDF["stator_vanes"] else "#ff9896" for v in VANES])
ax.set_xticks(range(len(VANES)), [str(v) for v in VANES]); ax.set(xlabel="stator vanes (12 blades)", ylabel="interaction tones, loudest direction [dB(A)]",
                                                                    title="the stator count (Tyler–Sofrin)")
ax.grid(color="#e8e8e8", lw=0.6, axis="y")
fig.tight_layout()
display(pd.DataFrame([{"vanes": r_["vanes"], "1×BPF cut off": r_["bpf_cut_off"], "harmonics that propagate": r_["cut_on_harmonics"],
                        "lowest |n| at 1×BPF": min(abs(m_["n"]) for m_ in r_["modes"] if m_["harmonic"] == 1)} for r_ in study]).set_index("vanes"))
LOUD_ALL = pd.DataFrame({**{f"open {c['label']}": {"BPF [Hz]": c["bpf_hz"], f"at {LISTEN_ANGLE:.0f}° [dB(A)]": c["dBA_listen"],
                                                  "disc plane [dB(A)]": c["dBA_disc_plane"],
                                                  "loudest direction [dB(A)]": max(c["directivity_dBA"]) if c["directivity_dBA"] else None}
                            for c in (PJ["configs"].values() if COMPARE else [])},
                         "EDF": {"BPF [Hz]": EDF_NOISE["bpf"], f"at {LISTEN_ANGLE:.0f}° [dB(A)]": EDF_NOISE["total_A"],
                                 "disc plane [dB(A)]": edf_noise(angle=90.0)["total_A"], "loudest direction [dB(A)]": float(EDF_DIRECTIVITY.max())}})
LOUD_ALL.round(1)

In [ ]:
# the EDF's sound; notebook 25's propeller sounds next to it (their own files, at notebook 25's common scale)
E_SIG = ap.synthesize(EDF_NOISE["tones"], EDF_NOISE["broadband"], seconds=3.0, rate=RATE, band=EDF_NOISE["band"], seed=9)
full_scale = (PJ["noise_setup"]["full_scale_pa"] if COMPARE else None) or 1.2 * float(np.abs(E_SIG).max())
full_scale = max(full_scale, 1.2 * float(np.abs(E_SIG).max()))
EDF_WAV = ap.write_wav(OUT / "sound_edf.wav", E_SIG, full_scale, RATE)
print(f"EDF at cruise: {EDF_NOISE['total_A']:.1f} dB(A) at {LISTEN_ANGLE:.0f}°, BPF {EDF_NOISE['bpf']:.0f} Hz — {EDF_WAV}")
display(Audio(url=str(EDF_WAV)))
for c in (PJ["configs"].values() if COMPARE else []):
    if c["wav"] and Path(c["wav"]).exists():
        print(f"open {c['label']} (notebook 25): {c['dBA_listen']:.1f} dB(A)")
        display(Audio(url=c["wav"]))

## 8. Vibration: the fan blade

One fan blade with its hub, held at the bore, spinning at the EDF's take-off rpm (`talos.Centrifugal`) and carrying its
share of the rotor's thrust and torque; its modes; a Campbell diagram with the orders that drive it — **1P** (unbalance),
**V·P** (the stators' potential field); and the unbalance force on the motor mount against the propellers'.

In [ ]:
eblade = PropellerCAD().generate(diameter=EDF["diameter_mm"], pitch=EDF["pitch_mm"], blades=1, hub_diameter=EDF["hub_diameter_mm"],
                                 hub_height=EDF["hub_height_mm"], bore=4.0, chord_root=EDF["chord_root_mm"], chord_max=EDF["chord_max_mm"],
                                 chord_tip=EDF["chord_tip_mm"], thickness=EDF["thickness"], camber=EDF["camber"], stations=6)
eb_step = eblade.export(RUNS / "cad_edf_blade", formats=("step",)).artifacts["step"]
Re_, hh_ = EDF["diameter_mm"] / 2, EDF["hub_height_mm"]
E_REGIONS = [talos.SurfacesInBox("bore", (-2.5, -2.5, -hh_ / 2 - 0.5, 2.5, 2.5, hh_ / 2 + 0.5)),
             talos.SurfacesInBox("blade", (EDF["hub_diameter_mm"] / 2 - 1.0, -Re_, -Re_, Re_ + 1.0, Re_, Re_))]
E_MESH = talos.MeshSettings(element_size=1.5, curvature_points=12, min_element_size=0.6)   # ~40 k elements; finer meshes run for many minutes
eto = EDF_OPS["take-off"]
Tb, Ft = eto.rotor_thrust / EDF["blades"], eto.torque / (EDF["blades"] * 0.7 * EDF_BLADE.radius)

def efea(name, loads):
    m = talos.StructuralModel(eb_step, "mm-N-MPa", BLADE_MAT, E_REGIONS, [talos.FixedSupport("bore")], loads, E_MESH, name=name)
    m.mesh(RUNS / f"fea_{name}")
    return m, m.solve(RUNS / f"fea_{name}")

em_c, er_c = efea("edf_takeoff", [talos.Centrifugal(eto.rpm, point=(0, 0, 0), axis=(0, 0, 1)), talos.Force("blade", fz=Tb, fy=-Ft)])
em_a, er_a = efea("edf_air", [talos.Force("blade", fz=1.0, fy=-Ft / Tb)])
shutil.copytree(RUNS / "fea_edf_air", RUNS / "fea_edf_modes", dirs_exist_ok=True)
e_modes = em_a.solve_modes(RUNS / "fea_edf_modes", n_modes=4)
print(er_c); print(e_modes)
ef0 = np.array(e_modes.metrics["frequencies_hz"])
tviz.show(tviz.plot_results(er_c, field="von_mises"))
EDF_STRESS_MOVIE = tviz.animate(er_c, OUT / "edf_blade_stress_take_off.mp4", rpm=eto.rpm, axis="z", seconds=4, fps=24)
display(Video(str(EDF_STRESS_MOVIE), embed=False, width=640))

In [ ]:
# Campbell: the fan blade against 1P, 2P and the stator orders; the stators' upstream potential field as the alternating load
V_ = EDF["stator_vanes"]
r_m = 0.5 * (EDF_BLADE.hub_radius + EDF_BLADE.radius)
pitch_v = 2 * math.pi * r_m / V_
gap_ = EDF["stator_gap_mm"] / 1000
blockage = EDF["stator_thickness"]
u_pot = (blockage * EDF["stator_chord_mm"] / 1000 / pitch_v) * math.exp(-2 * math.pi * gap_ / pitch_v)   # velocity ripple / V_fan at the rotor
rpm_ax = np.linspace(0, 1.1 * eto.rpm, 200)
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
for i, f in enumerate(ef0):
    ax[0].plot(rpm_ax, np.sqrt(f ** 2 + (1.2 if i == 0 else 0.0) * (rpm_ax / 60) ** 2), lw=2, label=f"mode {i + 1} ({f:.0f} Hz at rest)")
for q, lab in ((1, "1P"), (2, "2P"), (V_, f"{V_}P (stators)"), (2 * V_, f"{2 * V_}P"), (EDF["blades"], f"{EDF['blades']}P")):
    ax[0].plot(rpm_ax, q * rpm_ax / 60, color="#999999", lw=1.0 if q in (1, V_) else 0.7, ls="-" if q in (1, V_) else ":")
    ax[0].text(rpm_ax[-1], q * rpm_ax[-1] / 60, lab, fontsize=8, va="center")
ax[0].axvspan(EDF_OPS["cruise"].rpm, eto.rpm, color="#d62728", alpha=0.12, label="cruise .. take-off rpm")
ax[0].set(xlabel="rpm", ylabel="frequency [Hz]", title="Campbell diagram of the fan blade", ylim=(0, 1.15 * max(ef0.max(), 2 * V_ * rpm_ax[-1] / 60)))
ax[0].legend(fontsize=8); ax[0].grid(color="#e8e8e8", lw=0.6)
UNB = {}
for name, rpm_, m_rot in [(n, prop_op(n, "cruise")["rpm"], prop(n)["rotor_mass_kg"]) for n in PROP_NAMES] + [("EDF", EDF_OPS["cruise"].rpm, 0.060)]:
    UNB[name] = {g: boreas.unbalance_force(m_rot, rpm_, g) for g in (6.3, 2.5)}
x = np.arange(len(UNB))
ax[1].bar(x - 0.2, [v[6.3] for v in UNB.values()], 0.4, color=[UNIT_COLOR[k] for k in UNB], label="balance grade G 6.3")
ax[1].bar(x + 0.2, [v[2.5] for v in UNB.values()], 0.4, color="#cccccc", label="G 2.5")
ax[1].set_xticks(x, list(UNB)); ax[1].set(ylabel="rotating force at cruise [N] (1P)", title="unbalance: what the motor mount feels"); ax[1].legend(); ax[1].grid(color="#e8e8e8", lw=0.6, axis="y")
fig.tight_layout()

fr_c, fr_a = talos.read_frd(er_c.artifacts["frd"]), talos.read_frd(er_a.artifacts["frd"])
i_hot = int(np.argmax(fr_c.von_mises))
sm = float(fr_c.von_mises[i_hot])
sa = float(fr_a.von_mises[i_hot]) * Tb * 2 * u_pot                          # thrust ripple ~ 2 x velocity ripple (lift ~ U^2), at V P
fm_op = np.sqrt(ef0 ** 2 + np.array([1.2] + [0.0] * (len(ef0) - 1)) * (eto.rpm / 60) ** 2)
daf = max(1 / np.sqrt((1 - r_ ** 2) ** 2 + (2 * (MATERIAL["damping"] + 0.02) * r_) ** 2) for r_ in (V_ * eto.rpm / 60 / fm_op))
sf_good = 1 / (sa * daf / MATERIAL["fatigue_1e7"] + sm / MATERIAL["ultimate"])
margins = {q: min(abs(f - q * EDF_OPS["cruise"].rpm / 60) / f for f in np.sqrt(ef0 ** 2 + np.array([1.2] + [0.0] * (len(ef0) - 1)) * (EDF_OPS["cruise"].rpm / 60) ** 2))
           for q in (1, 2, V_, 2 * V_)}
EDF_VIB = pd.Series({"max von Mises at take-off [MPa]": sm, "SF yield, take-off": MATERIAL["yield_"] / sm,
                     f"stator potential-field ripple at the rotor (u/V_fan, {V_}P)": u_pot, "alternating stress [MPa]": sa * daf,
                     "Goodman SF, take-off": sf_good, "frequency margin at cruise (1P, 2P, VP, 2VP)": ", ".join(f"{v:.2f}" for v in margins.values())},
                    name="EDF fan blade")
EDF_VIB.to_frame()

## 9. Side by side

In [ ]:
def pj_cfg(name, layout, key):
    v = PJ["configs"][f"{layout} {name.split()[0]}"][key] if COMPARE else None
    return np.nan if v is None else v

def loudest(name, layout):
    d_ = PJ["configs"][f"{layout} {name.split()[0]}"]["directivity_dBA"] if COMPARE else None
    return max(d_) if d_ else np.nan

SIDE = {}
for name in PROP_NAMES:
    SIDE[f"open {name}"] = {
        "η cruise": prop_op(name, "cruise")["efficiency"], "cruise shaft power [W]": POWER.loc["cruise shaft power [W]", name],
        "take-off shaft power [W]": POWER.loc["take-off shaft power [W]", name], "endurance [min]": POWER.loc["endurance at cruise [min]", name],
        "time 9 → 20 m/s [s]": ACC_TAB.loc["time 9 → 20 m/s [s]", name], "top speed [m/s]": ACC_TAB.loc["top speed, level [m/s]", name],
        "best climb [m/s]": ACC_TAB.loc["best climb rate [m/s]", name],
        "loudest dB(A) at 10 m (tractor / pusher)": f"{loudest(name, 'tractor'):.1f} / {loudest(name, 'pusher'):.1f}",
        "BPF [Hz]": pj_cfg(name, "tractor", "bpf_hz"), "blade SF yield (take-off)": prop(name)["blade_sf_yield_take_off"] or np.nan,
        "Goodman SF (climb, tractor / pusher)": f"{pj_cfg(name, 'tractor', 'goodman_sf_climb'):.1f} / {pj_cfg(name, 'pusher', 'goodman_sf_climb'):.1f}",
        "unbalance at cruise, G 6.3 [N]": UNB[name][6.3]}
SIDE[f"EDF {EDF['diameter_mm']:.0f} mm, {EDF['blades']} blades"] = {
    "η cruise": float(np.interp(POINTS["cruise"][0], VS, ETA_LEVEL["EDF"])), "cruise shaft power [W]": POWER.loc["cruise shaft power [W]", "EDF"],
    "take-off shaft power [W]": POWER.loc["take-off shaft power [W]", "EDF"], "endurance [min]": POWER.loc["endurance at cruise [min]", "EDF"],
    "time 9 → 20 m/s [s]": ACC_TAB.loc["time 9 → 20 m/s [s]", "EDF"], "top speed [m/s]": ACC_TAB.loc["top speed, level [m/s]", "EDF"],
    "best climb [m/s]": ACC_TAB.loc["best climb rate [m/s]", "EDF"],
    "loudest dB(A) at 10 m (tractor / pusher)": f"{EDF_DIRECTIVITY.max():.1f} (no installation)", "BPF [Hz]": EDF_NOISE["bpf"],
    "blade SF yield (take-off)": EDF_VIB["SF yield, take-off"], "Goodman SF (climb, tractor / pusher)": f"{EDF_VIB['Goodman SF, take-off']:.1f} (take-off)",
    "unbalance at cruise, G 6.3 [N]": UNB["EDF"][6.3]}
SIDE = pd.DataFrame(SIDE)
display(SIDE)
names = PROP_NAMES + ["EDF"]
fig, axs = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (key, vals, better) in zip(axs, (
        ("η at cruise", [ETA_LEVEL[n][np.argmin(abs(VS - POINTS['cruise'][0]))] for n in names], "higher"),
        ("endurance at cruise [min]", [POWER.loc["endurance at cruise [min]", n] for n in names], "higher"),
        ("time 9 → 20 m/s [s]", [ACC_TAB.loc["time 9 → 20 m/s [s]", n] for n in names], "lower"),
        ("top speed [m/s]", [ACC_TAB.loc["top speed, level [m/s]", n] for n in names], "higher"))):
    ax.bar(range(len(names)), vals, color=[UNIT_COLOR[n] for n in names])
    ax.set_xticks(range(len(names)), [("open " + n.split()[0]) if n != "EDF" else "EDF" for n in names]); ax.set_title(f"{key}\n({better} is better)", fontsize=10)
    ax.grid(color="#e8e8e8", lw=0.6, axis="y")
fig.tight_layout()
(RUNS / "edf.json").write_text(json.dumps({"edf": EDF, "housing": {k: v for k, v in HG.items() if np.isscalar(v)}, "compared_with": "notebook 25" if COMPARE else None,
    "operating_points": {k: v.to_dict() for k, v in EDF_OPS.items()}, "power": POWER.to_dict(), "acceleration": ACC_TAB.to_dict(),
    "noise": LOUD_ALL.to_dict(), "vibration": EDF_VIB.to_dict(), "side_by_side": SIDE.astype(str).to_dict(),
    "cfd": {"preset": CFD_PRESET, "cases": {k: {kk: vv for kk, vv in r.metrics.items() if not isinstance(vv, (list, dict))} for k, r in EDF_CFD.items()}}},
    indent=2, default=float))
print("written", RUNS / "edf.json")

**Reading the comparison.** The ducted fan does the same job with a much smaller disc, so it throws a small, fast jet:
at the drone's speeds its propulsive efficiency is low and it needs more shaft power at cruise and at take-off — the
endurance on the same battery falls. What it buys: thrust that holds up with speed (better acceleration at the top end,
a higher top speed), blades shielded inside a duct (safety, and the rotor-alone tones cut off), and a compact unit. It
is not quiet: twelve blades at its rpm put the blade-passing tone in the kilohertz range where the A-weighting counts
it fully, and the rotor–stator interaction tones escape when the stator count lets a low-order mode through — the
stator-count bar shows which counts keep the first harmonic in the duct. Vibration: the fan blade is short and stiff,
its modes sit far above 1P; the excitations to watch are the stator orders (V·P) and the unbalance, which grows with
the square of the rpm.

## 10. Multi-stage ducted fans — two and three fan rings

A **multi-stage fan** has two or three **fan rings** — rotors — one behind the other on one shaft in the same duct, each
followed by its ring of stator vanes (`boreas.ducted`, `stages`; two stages = two fan rings and two stator rings). The
stator straightens the flow behind each rotor, so every rotor meets the same axial inflow at the same flow speed and adds
the same pressure rise: the stages add their rise (and take the same power each) at the **same mass flow**. Each stage
after the first lengthens the duct (outside friction) and adds an interstage loss (`interstage_loss`, 0.03 dynamic
pressures: the stator's wake and the gap in front of the next rotor); its rotor and stator weigh about 45 g at 90 mm.

What that buys: more pressure — a faster jet — **at the same rpm**. What it does not: the jet is still the same small
area, so at the same thrust it is no more efficient (it is a little less: more blades, more wetted duct). A second
stage therefore pays only when a single stage **runs out of rpm before it runs out of power**. At the 240 m/s tip speed of
hobby fans that takes a lot of power in a small duct (a 70 mm fan absorbs about 3.8 kW of shaft power in one stage before
its tips reach it), but it is exactly the case of a **quiet fan**, held to a low tip speed for noise (the blade-passing
tones fall steeply with tip speed). Below: the 90 mm fan of section 13 with one, two and three stages at the catalogue
power, then quiet 70 and 90 mm fans held to 160 m/s at the tips, at the same power.

In [ ]:
from dataclasses import replace as _replace
STAGES = (1, 2, 3)
STAGE_COLOR = {1: "#d62728", 2: "#ff9896", 3: "#8c1c13"}   # ducted fans in reds
P_SHAFT = EDF_STATIC_POINT["electrical_W"] * EDF_STATIC_POINT["drive_efficiency"]      # the catalogue fan's full shaft power
TIP_SPEED = 240.0                                                                         # m/s: the fans' rpm limit

def full_power_point(fan, V, p_shaft, rpm_max):
    """The fan at airspeed V at full power: the rpm where the shaft power reaches p_shaft, or rpm_max (tip speed) first."""
    lo, hi = 2000.0, rpm_max
    if ducted.solve(fan, SECTION, hi, V, RHO).power <= p_shaft:
        return ducted.solve(fan, SECTION, hi, V, RHO)
    for _ in range(24):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if ducted.solve(fan, SECTION, mid, V, RHO).power <= p_shaft else (lo, mid)
    return ducted.solve(fan, SECTION, lo, V, RHO)

VS14 = np.linspace(0.0, 70.0, 15)
fig, ax = plt.subplots(1, 3, figsize=(20, 4.6))
STAGE_TAB = {}
for n in STAGES:
    fan_n = _replace(FAN, stages=n)
    rpm_max = TIP_SPEED / (math.pi * EDF_BLADE.diameter) * 60
    pts = [full_power_point(fan_n, V, P_SHAFT, rpm_max) for V in VS14]
    net = np.array([p.thrust - ducted.nacelle_drag(fan_n, V, RHO, NU) for p, V in zip(pts, VS14)])
    ax[0].plot(VS14, net, color=STAGE_COLOR[n], lw=2, label=f"{n} stage{'s' if n > 1 else ''}")
    ax[1].plot(VS14, [p.rpm for p in pts], color=STAGE_COLOR[n], lw=2, label=f"{n}")
    ax[2].plot(VS14, [p.exit_velocity for p in pts], color=STAGE_COLOR[n], lw=2, label=f"{n}")
    lvl = [edf_point(V, required_thrust(V)) if n == 1 else ducted.rpm_for_thrust(fan_n, SECTION, required_thrust(V) + ducted.nacelle_drag(fan_n, V, RHO, NU), V, RHO)
           for V in (15.0, 25.0)]
    STAGE_TAB[f"90 mm, {n} stage{'s' if n > 1 else ''}"] = {"static thrust at full power [N]": net[0], "thrust at 40 m/s [N]": float(np.interp(40, VS14, net)),
                                                       "rpm at full power, static": pts[0].rpm, "jet speed static [m/s]": pts[0].exit_velocity,
                                                       "shaft power, level flight 15 m/s [W]": lvl[0].power, "shaft power, level flight 25 m/s [W]": lvl[1].power}
ax[1].axhline(TIP_SPEED / (math.pi * EDF_BLADE.diameter) * 60, color="#999999", ls=":", label="tip-speed limit")
ax[0].set(xlabel="airspeed [m/s]", ylabel="net thrust [N]", title=f"90 mm fan at full power ({P_SHAFT:.0f} W shaft)"); ax[0].legend()
ax[1].set(xlabel="airspeed [m/s]", ylabel="rpm", title="the rpm it runs at"); ax[1].legend(title="stages", fontsize=8)
ax[2].set(xlabel="airspeed [m/s]", ylabel="m/s", title="jet speed at the nozzle"); ax[2].legend(title="stages", fontsize=8)
for a in ax:
    a.grid(color="#e8e8e8", lw=0.6)
fig.tight_layout()

In [ ]:
# where a second stage pays: quiet fans held to 160 m/s at the tips, at the catalogue power
import propulsor_maps as pmaps
QUIET_TIP = 160.0
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for D, ls in ((70.0, "-"), (90.0, "--")):
    for n in STAGES:
        fq, hq, _ = pmaps.fan_object(D, 1.78, 0.8, FAN.duct_loss, n)
        rq = QUIET_TIP / (math.pi * hq["diameter"] / 1000) * 60
        pts = [full_power_point(fq, V, P_SHAFT, rq) for V in VS14]
        net = np.array([p.thrust - ducted.nacelle_drag(fq, V, RHO, NU) for p, V in zip(pts, VS14)])
        ax[0].plot(VS14, net, ls, color=STAGE_COLOR[n], lw=2, label=f"{D:.0f} mm, {n} stage{'s' if n > 1 else ''}")
        ax[1].plot(VS14, [p.power for p in pts], ls, color=STAGE_COLOR[n], lw=2, label=f"{D:.0f} mm, {n}")
        STAGE_TAB[f"quiet {D:.0f} mm (tip {QUIET_TIP:.0f} m/s), {n} stage{'s' if n > 1 else ''}"] = {
            "static thrust at full power [N]": net[0], "thrust at 40 m/s [N]": float(np.interp(40, VS14, net)),
            "rpm at full power, static": pts[0].rpm, "jet speed static [m/s]": pts[0].exit_velocity, "shaft power absorbed, static [W]": pts[0].power}
ax[1].axhline(P_SHAFT, color="#999999", ls=":", label="power available")
ax[0].set(xlabel="airspeed [m/s]", ylabel="net thrust [N]", title=f"quiet fans: tips held to {QUIET_TIP:.0f} m/s, {P_SHAFT:.0f} W shaft available"); ax[0].legend(fontsize=8)
ax[1].set(xlabel="airspeed [m/s]", ylabel="shaft power [W]", title="the power it can absorb"); ax[1].legend(title="stages", fontsize=8)
for a in ax:
    a.grid(color="#e8e8e8", lw=0.6)
fig.tight_layout()
pd.DataFrame(STAGE_TAB).round(1)

**Reading the stages.** At the catalogue power the 90 mm fan is **power-limited** with one stage: a second stage makes
the same shaft power at a lower rpm and throws a faster jet, so its thrust is no higher and its level-flight power is
higher (less propulsive efficiency, more friction). The quiet 70 mm fan is the opposite case: held to 160 m/s at the tips,
one stage absorbs only about two thirds of the power; a second stage takes all of it and gives about a quarter more
thrust (a third adds nothing: the second already reached the power). The quiet 90 mm fan absorbs the power in one stage,
so its stages lose again. A multi-stage fan is the way to put more power through a small or slow duct — compactness and
noise — not the way to make a duct efficient, and not a way to make MERLIN faster: the race (notebook 27) has the stages in its design space.

## 11. The fan maps for the flight models

MERLIN's mission and race (notebooks 26b and 27) read maps instead of solving fans. Here every fan of a design space — 70–120 mm
(this fan scaled: hub, chords, stators, lip and duct), blade pitch 1.4–2.2 diameters, nozzle exit 65–90 % of the fan annulus,
the catalogue loss (fitted above) or a well-made duct (0.20), one to three stages — plus the lossless bound is solved over
airspeed × rpm and written to `designs/data/edf_maps.json` (`designs/propulsor_maps.py`; about 10 minutes on four cores).
The committed file is loaded unless `AIR_PROP_MAPS=build`.

In [ ]:
import propulsor_maps as pmaps
if os.environ.get("AIR_PROP_MAPS") == "build" or not pmaps.LIBRARIES["edf"].exists():
    print("written", pmaps.save(pmaps.build_library("edf", progress=True)))
FMAPS = pmaps.load(pmaps.LIBRARIES["edf"])
fans = [e for e in FMAPS["entries"] if "error" not in e]
rows = {}
for e in fans:
    u = pmaps.unit(e, 1900.0)
    rows[e["label"]] = {"stages": e["stages"], "quality": e["quality"], "static thrust at 1900 W [N]": u.full(0.0)[0],
                        "at 30 m/s [N]": u.full(30.0)[0], "at 60 m/s [N]": u.full(60.0)[0]}
FAN_TAB = pd.DataFrame(rows).T
fig, ax = plt.subplots(figsize=(8, 4.4))
for n, col in ((1, "#d62728"), (2, "#ff9896"), (3, "#8c1c13")):
    s_ = FAN_TAB[(FAN_TAB["stages"] == n) & (FAN_TAB["quality"] != "lossless")]
    ax.scatter(s_["static thrust at 1900 W [N]"], s_["at 60 m/s [N]"], s=14, color=col, label=f"{n} stage{'s' if n > 1 else ''}")
s_ = FAN_TAB[FAN_TAB["quality"] == "lossless"]
ax.scatter(s_["static thrust at 1900 W [N]"], s_["at 60 m/s [N]"], s=22, marker="x", color="k", label="lossless bound")
ax.set(xlabel="static thrust at 1900 W [N]", ylabel="thrust at 60 m/s [N]", title=f"{len(fans)} fan maps at 1900 W electrical"); ax.legend(fontsize=8)
ax.grid(color="#e8e8e8", lw=0.6)
FAN_TAB.sort_values("at 60 m/s [N]", ascending=False).head(12).round(2)